In [1]:
import os
import requests
import time
import pandas as pd

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [2]:
DATA_FOLDER = "./f1_data"
API_URL = "https://api.openf1.org/v1"

os.makedirs(DATA_FOLDER, exist_ok=True)

requests_session = requests.Session()

# Configure the retry strategy
retry_strategy = Retry(
    total=5,  # Maximum number of retries before giving up
    backoff_factor=1,  # Wait times: 1s, 2s, 4s, 8s, 16s between retries
    status_forcelist=[429, 500, 502, 503, 504], # HTTP status codes to retry on
    allowed_methods=["GET"]
)

# Mount the strategy to both HTTP and HTTPS
adapter = HTTPAdapter(max_retries=retry_strategy)
requests_session.mount("http://", adapter)
requests_session.mount("https://", adapter)

In [3]:
def fetch(endpoint, params=None):
    try:
        response = requests_session.get(f"{API_URL}/{endpoint}", params=params, timeout=15)
        response.raise_for_status()
        data = response.json()
        return data
    except Exception as e:
        print(e)
        return []

In [4]:
def save_to_csv(data, folder, filename):
    if not data:
        return
    df = pd.DataFrame(data)
    path = os.path.join(folder, f"{filename}.csv")
    df.to_csv(path, index=False)

In [5]:
def collect_sessions(session_key, session_name):
    race_folder = os.path.join(DATA_FOLDER, session_name.replace(' ', '_'))
    os.makedirs(race_folder, exist_ok=True)

    endpoints = {
        'weather': 'weather',
        'drivers': 'drivers',
        'stints': 'stints',
        'laps': 'laps',
        'pits': 'pit',
        'results': 'session_result'
    }

    for key, endpoint in endpoints.items():
        time.sleep(1)
        data = fetch(endpoint, {'session_key': session_key})
        save_to_csv(data, race_folder, key)

In [6]:
# 1. Get the list of races
print("Getting race list...")
all_sessions = []
for year in [2024, 2025, 2026]:
    time.sleep(0.5)
    sessions = fetch('sessions', {'year': year, 'session_type': 'Race'})
    all_sessions.extend(sessions)

print(f"Found {len(all_sessions)} races\n")

# 2. Process each race
for session in all_sessions:
    time.sleep(0.5)
    session_key = session['session_key']
    name = f"{session['year']}_{session['country_name']}_{session['session_name']}"

    # Simple check: if the folder exists, we've likely processed it
    if os.path.exists(os.path.join(DATA_FOLDER, name.replace(' ', '_'))):
        print(f"Skipping: {name} (folder exists)")
        continue

    collect_sessions(session_key, name)
    time.sleep(0.5)

Getting race list...
Found 90 races

404 Client Error: Not Found for url: https://api.openf1.org/v1/pit?session_key=9549
Skipping: 2024_Italy_Race (folder exists)
Skipping: 2024_United States_Sprint (folder exists)
Skipping: 2024_United States_Race (folder exists)
404 Client Error: Not Found for url: https://api.openf1.org/v1/pit?session_key=9635
Skipping: 2024_United States_Race (folder exists)
404 Client Error: Not Found for url: https://api.openf1.org/v1/pit?session_key=9934
Skipping: 2025_Italy_Race (folder exists)
Skipping: 2025_United States_Sprint (folder exists)
Skipping: 2025_United States_Race (folder exists)
Skipping: 2025_United States_Race (folder exists)
404 Client Error: Not Found for url: https://api.openf1.org/v1/weather?session_key=11261
404 Client Error: Not Found for url: https://api.openf1.org/v1/stints?session_key=11261
404 Client Error: Not Found for url: https://api.openf1.org/v1/laps?session_key=11261
404 Client Error: Not Found for url: https://api.openf1.org/